So I want to interact with the derived class in almost the same way that I interact with the `numpy.random.Generator` class.  How does one usually interact with the `Generator` class??

Usually, one calls the `default_rng(seed)` function to instantiate the `Generator` instance.  This means that I'll want a similar function to preferentially initiate my derived class.  we might call that function `prime_rng()`

Then, we use methods from the `Generator` object to generate random numbers.  So we'll need to add new methods to our derived class.  

Luckily, all the methods from the base class can be inherited pretty easily in Python.  Sadly, the random module is not written in pure Python, it is written in Cython, which is Python with C added on top.  I have NO CLUE how inheritance works in Cython.  Also, I don't really understand how `default_rng()` works, despite looking over the numpy source code.  So we have a serious problem.  

I suppose I need to figure out the `__init__(self)` function for my derived class first.  In numpy, the `__init__()` function for the `Generator` class does not have all the fancy functionality that the `default_rng()` function implements.  I hope there is a reason that the class is set up in this way.  I will follow their example.  

Good news! from the Cython Docs:

*"Normal Python classes can inherit from cdef classes, but not the other way around."*

So we will do our inheritance in pure python.  Although that might make the `__init__()` function a bit harder to write

In [1]:
import numpy as np 
from numpy.random._pcg64 import PCG64

Honestly a good first goal is just to attempt to make a derived class without any additional methods.  Just make sure that the `__init__()` function works properly.  That way we don't have to worry about anything else.

I don't think I'll need any special init behavior, so I should be able to just make my init function a thin wrapper around `Generator.__init__()`

In [2]:
class GeneratorWrapper(np.random.Generator):
    """Test class to make sure I remember how class definitions work."""

    def __init__(self, bit_generator):
        np.random.Generator.__init__(self, bit_generator)

In [3]:
seeda = 137
rnga = GeneratorWrapper(PCG64(seeda))

In [4]:
rnga.random()

0.8581820515662284

In [5]:
type(rnga)

__main__.GeneratorWrapper

In [7]:
print(str(rnga._bit_generator))

In [6]:
# This code was copied from ljs on StackOverflow

rnga_methods = [method_name for method_name in dir(object)
                  if callable(getattr(rnga, method_name))]

for method in rnga_methods:
    print(str(method))


__class__
__delattr__
__dir__
__eq__
__format__
__ge__
__getattribute__
__getstate__
__gt__
__hash__
__init__
__init_subclass__
__le__
__lt__
__ne__
__new__
__reduce__
__reduce_ex__
__repr__
__setattr__
__sizeof__
__str__
__subclasshook__


hmmmmmmmmmmmm.  fuck.

But the `rnga.random()` worked just fine.  so the inheritance seems to work just fine.  

theoretically, I have created a derived class.  Now I want to make a method that implements all the fancy shit `default_rng()` does.  

In [11]:
def wrapper_rng(seed=None):
    if isinstance(seed, np.random.BitGenerator):
        # We were passed a BitGenerator, so just wrap it up.
        return GeneratorWrapper(seed)
    elif isinstance(seed, np.random.Generator):
        # We were passed a Generator, so wrap up its BitGenerator.   
        return GeneratorWrapper(seed._bit_generator)
    elif isinstance(seed, np.random.RandomState):
        gen = GeneratorWrapper(seed._bit_generator)
        return gen

    # Otherwise we need to instantiate a new BitGenerator and Generator as
    # normal.
    return GeneratorWrapper(PCG64(seed))


In [12]:
rng = np.random.default_rng(31415)

rngb = wrapper_rng(rng)

rngb.random()

0.49968932952827405

Okay I think this will work fine.  I have no idea how I'm going to actually write this stuff out but its probably fine.  